In [1]:
# Load required libraries
library(arrow)
library(broom)
library(dplyr)
library(mgcv)
library(ggplot2)
library(lubridate)
library(parallel)
library(mlflow)
library(plotly)
library(gratia)
# Determine the number of cores to use (leave one free for the OS)
num_cores <- detectCores() - 1

# Create a cluster
# cl <- makeCluster(num_cores)


Attaching package: ‘arrow’


The following object is masked from ‘package:utils’:

    timestamp



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: nlme


Attaching package: ‘nlme’


The following object is masked from ‘package:dplyr’:

    collapse


This is mgcv 1.9-4. For overview type '?mgcv'.


Attaching package: ‘lubridate’


The following object is masked from ‘package:arrow’:

    duration


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union



Attaching package: ‘plotly’


The following object is masked from ‘package:ggplot2’:

    last_plot


The following object is masked from ‘package:arrow’:

    schema


The following object is masked from ‘package:stats’:

    filter


The following object is masked from ‘package:graphics’:

    layout




In [2]:
mlflow_set_tracking_uri("https://mlflow.analytics-dev.tapestry.com")

In [ ]:
exp <- mlflow_set_experiment("zipline_smart_associate_goals")

In [ ]:
# Load the data
associate_goals_df <- arrow::read_feather("../data/associate_goals_df.feather")


In [ ]:
head(associate_goals_df)

In [ ]:
sample_doors <- associate_goals_df %>%
  filter(!is.na(CHNL_DESC)) %>%
  group_by(CHNL_DESC) %>%
  distinct(STORE_NUMBER) %>% 
  sample_n(50)



In [ ]:
colnames(associate_goals_df)

In [ ]:
associate_goals_df %>% 
    head(10)

In [ ]:
regression_df <- associate_goals_df %>%
       # Filter for valid goal hours and sales amounts
       filter(!is.na(GOALS_SCHEDULED_HOURS_CORRECTED), 
                             GOALS_SCHEDULED_HOURS_CORRECTED >= 4,
                             GOALS_SCHEDULED_HOURS_CORRECTED <= 9,
                             # GROSS_SALES_AMOUNT_DLABS >= 100,
                             GROSS_SALES_AMOUNT_DLABS < quantile(associate_goals_df$GROSS_SALES_AMOUNT_DLABS, 0.99999, na.rm = TRUE), 
                             STORE_MANAGER_GOAL > 0, 
                             !is.na(HOURS_SINCE_OPEN)) %>% 
       # Filter for sampled stores
       # filter(STORE_NUMBER %in% sample_doors$STORE_NUMBER) %>%
       # Create derived variables
       mutate(YEAR = year(BUSINESS_DATE), 
                             SEASONAL_HIRE = ifelse(month(FIRST_SELLING_DAY) %in% c(11, 12), 1, 0),
                             IS_HOLIDAY_SEASON = ifelse(MONTH %in% c(11, 12), 1, 0),
                             IS_WEEKEND = ifelse(DAY_OF_WEEK %in% c(0, 5, 6), 1, 0),
                             SALES_PER_GOAL_HOUR = GROSS_SALES_AMOUNT_DLABS / GOALS_SCHEDULED_HOURS_CORRECTED,
                             LOG_TENURE = log(TENURE_MONTHS + 1),
                             LOG_SALES = log(GROSS_SALES_AMOUNT_DLABS + 1)) %>%
       # Select relevant columns
       select(SALES_ASSOCIATE, STORE_NUMBER, GROSS_SALES_AMOUNT_DLABS, STORE_MANAGER_GOAL, 
                             MONTH, DAY_OF_WEEK, WEEK_OF_YEAR, YEAR, TENURE_MONTHS, 
                             GOALS_SCHEDULED_HOURS_CORRECTED, SEASONAL_HIRE, IS_WEEKEND, IS_HOLIDAY_SEASON,
                             CHNL_DESC, REGION_DESC, DISTRICT_DESC, SALES_PER_GOAL_HOUR, LOG_TENURE, LOG_SALES, HOURS_SINCE_OPEN)

In [ ]:

# Interactive histogram for Goals Scheduled Hours
p1 <- ggplot(regression_df, aes(x = GOALS_SCHEDULED_HOURS_CORRECTED)) +
    geom_histogram(bins = 50, fill = "steelblue", color = "white") +
    labs(title = "Distribution of Goals Scheduled Hours",
         x = "Goals Scheduled Hours",
         y = "Frequency") +
    theme_minimal()

ggplotly(p1)

# Interactive histogram for Log Sales
p2 <- ggplot(regression_df, aes(x = GROSS_SALES_AMOUNT_DLABS)) +
    geom_histogram(bins = 50, fill = "steelblue", color = "white") +
    labs(title = "Distribution of Log Sales",
         x = "Log Sales",
         y = "Frequency") +
    theme_minimal()
    
ggplotly(p2)


In [ ]:
p3 <- ggplot(regression_df, aes(x = SALES_PER_GOAL_HOUR)) +
    geom_histogram(bins = 50, fill = "steelblue", color = "white") +
    labs(title = "Distribution of Sales Per Goal Hour",
             x = "Sales Per Goal Hour",
             y = "Frequency") +
    theme_minimal() 

ggplotly(p3)

In [ ]:
summary(regression_df$SALES_PER_GOAL_HOUR)

In [ ]:
head(regression_df)

In [ ]:
regression_df <- regression_df %>%
    mutate(across(c(STORE_NUMBER, SALES_ASSOCIATE, MONTH, IS_WEEKEND, IS_HOLIDAY_SEASON,
                                    WEEK_OF_YEAR, YEAR, SEASONAL_HIRE, DAY_OF_WEEK, CHNL_DESC, 
                                    REGION_DESC, DISTRICT_DESC), 
                                as.factor))


In [ ]:
summary(regression_df)

This model accounts for many possible scenarios: 
1. That sales associates that join have various skills and previous experiences so we allow each associate to have their own intercept. 
2. Sales Associates pick up at different rates, we allow separate slopes per associate
3. We control for Stores, and the Time of sales (Day of week, Week of Year)
4. Associates with Short Tenure might not get scheduled on prime days so there may be an interaction effect here. 
5. The Seasonal associates hired in Holiday might be different than others so we include the MONTH hired. 
6. The effect of Tenure may diminish over time - i.e. the difference between 1 month an 6 months is significantly more than 24 months at 32 months. 
7. The effect of Tenure is not linear, it is likely a curve. 

In [ ]:

log_model <- function(formula, model, experiment_id) {
    
    with(mlflow_start_run(experiment_id = experiment_id), {
        # Log parameters
        mlflow_log_param("formula", paste(deparse(formula), collapse = " "))
        mlflow_log_param("model_type", "mgcv::bam")

        # Extract k_value from formula (if present)
        formula_str <- paste(deparse(formula), collapse = " ")
        k_match <- regexpr("k\\s*=\\s*[0-9]+", formula_str)
        if (k_match > 0) {
            k_value <- gsub(".*k\\s*=\\s*([0-9]+).*", "\\1", regmatches(formula_str, k_match))
            mlflow_log_param("k_value", k_value)
        }

        # Extract smooth terms from formula
        smooth_pattern <- "s\\([^)]+\\)"
        smooth_matches <- gregexpr(smooth_pattern, formula_str)
        smooth_terms <- regmatches(formula_str, smooth_matches)[[1]]
        if (length(smooth_terms) > 0) {
            mlflow_log_param("smooth_terms", paste(smooth_terms, collapse = ", "))
        }
        
        # Get model summary metrics
        summ <- broom::glance(model)
        mlflow_log_metric("AIC", summ$AIC)
        mlflow_log_metric("adj.r.squared", summ$adj.r.squared)
        mlflow_log_metric("deviance", summ$deviance)
    
        # Create a PDF file with diagnostic plots
        pdf("model_diagnostics.pdf", width = 11, height = 8.5)
        gam.check(model)
        dev.off()
        mlflow_log_artifact("model_diagnostics.pdf")
        file.remove("model_diagnostics.pdf")
    
        # Log model fit plots
        pdf("model_fits.pdf", width = 11, height = 8.5)
        plot(model, shade = TRUE, seWithMean = TRUE, rug = TRUE)
        dev.off()
        mlflow_log_artifact("model_fits.pdf")
        file.remove("model_fits.pdf")
        
        # Save model as RDS file and log as artifact
        saveRDS(model, "model.rds")
        mlflow_log_artifact("model.rds")
        file.remove("model.rds")
    })

}


In [ ]:
form <- as.formula('SALES_PER_GOAL_HOUR ~ s(LOG_TENURE, k = 4) + s(STORE_NUMBER, bs = "re") + s(SALES_ASSOCIATE, bs = "re") + s(HOURS_SINCE_OPEN, k = 4) + DAY_OF_WEEK + STORE_MANAGER_GOAL + IS_HOLIDAY_SEASON')
fitted_model <- bam(form, data = regression_df, nthreads = num_cores, discrete = TRUE, family = tw(link = "log")) # Use the Tweedie family))
log_model(form, fitted_model, experiment_id = exp)

In [ ]:
concurvity(fitted_model, full = FALSE)

In [ ]:
summary(fitted_model)

In [ ]:
appraise(fitted_model)

There doesn't really appear to be an interaction between the day of the week scheduled and tenure once we add the sales manager goals - let's simplify the model

The spline isn't doing much here, lets simplify even more. 

In [ ]:
draw(fitted_model)

In [ ]:
best_fit <- mlflow_download_artifacts(run_id = '420bf1d81fcc415da966bd4e905decbd',path = "model.rds")

In [ ]:
best_fit <- readRDS(best_fit)

In [ ]:
draw(best_fit)

In [ ]:
summary(best_fit)

In [ ]:
anova(best_fit)